# Paper figures

Regenerates the result figures of the paper from the released per-citation outputs in `results/` (no API calls):

* Figure 3: NeurIPS 2025 status distribution
* Figure 4: per-field score distributions on NeurIPS 2025
* Figure 5: CiteAudit status distribution
* Figure 6: materials references by type

Figures 1 (pipeline diagram) and 2 (web-tool screenshot) are not data plots; they are included in `figures/`.

In [ ]:
import os, sys, json
from collections import Counter

if "google.colab" in sys.modules:
    if not os.path.exists("hallucination_detector"):
        !git clone -q https://github.com/atomgptlab/hallucination_detector.git
    %cd hallucination_detector
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

import matplotlib.pyplot as plt

OUT = "figures"
os.makedirs(OUT, exist_ok=True)
COLORS = {"likely hallucinated": "#FF4B6E", "partial": "#FFB52E", "verified": "#2EC4B6"}
ORDER = ["likely hallucinated", "partial", "verified"]

def load(path):
    with open(path) as f:
        return json.load(f)["results"]

neurips = load("results/neurips2025_v1.json")
citeaudit = load("results/citeaudit_300_v2.json")
materials = load("results/materials_pilot_v2.json")
print(len(neurips), len(citeaudit), len(materials))

In [ ]:
def status_bar(results, title, fname):
    counts = Counter(r["status"] for r in results)
    vals = [counts[s] for s in ORDER]
    fig, ax = plt.subplots(figsize=(6.1, 4.9))
    bars = ax.bar(ORDER, vals, color=[COLORS[s] for s in ORDER], width=0.7)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.01, str(v),
                ha="center", va="bottom", fontsize=13, fontweight="bold")
    ax.set_ylabel("Number of Citations")
    ax.set_ylim(0, max(vals) * 1.15)
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, fname), dpi=100)
    plt.show()
    return dict(zip(ORDER, vals))

# Figure 3
print(status_bar(neurips, "AtomGPT: Status Distribution\n(100 Known Hallucinated Citations, NeurIPS 2025)",
                 "ui_status_distribution.png"))

In [ ]:
# Figure 4: per-field scores over the citations where each field was extracted
fields = [("title_score", "Title match"), ("author_score", "Author match"), ("journal_score", "Journal match")]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (key, name), lab in zip(axes, fields, "abc"):
    vals = [r[key] for r in neurips if r[key] is not None]
    mean = sum(vals) / len(vals)
    ax.hist(vals, bins=20, range=(0, 100), color="#4C72B0", edgecolor="white")
    ax.axvline(mean, color="#C44E52", ls="--", label=f"mean {mean:.0f}")
    ax.set_title(f"({lab}) {name}  (n={len(vals)})")
    ax.set_xlabel("Score (0-100)")
    ax.set_ylabel("Citations")
    ax.legend()
    print(name, len(vals), round(mean, 1))
plt.tight_layout()
plt.savefig(os.path.join(OUT, "ui_component_panels.pdf"))
plt.show()

In [ ]:
# Figure 5
print(status_bar(citeaudit, "AtomGPT Hallucination Detector: Status Distribution\n"
                 "(299 CiteAudit Citations: 149 genuine + 150 fabricated)",
                 "atomgpt_citeaudit_300.png"))

In [ ]:
# Figure 6: genuine materials references by type
cats = [("A_regular_article", "Regular\narticles"), ("B_database_infra", "Database /\ninfrastructure"),
        ("C_software", "Software"), ("D_crystallographic_db", "Crystallographic\nDB"),
        ("E_bare_data_entry", "Bare data /\nsoftware entries")]
counts = {c: Counter(r["status"] for r in materials if r["category"] == c) for c, _ in cats}
fig, ax = plt.subplots(figsize=(11.8, 5.8))
x = range(len(cats))
bottom = [0] * len(cats)
for s in ["verified", "partial", "likely hallucinated"]:
    h = [counts[c][s] for c, _ in cats]
    ax.bar(x, h, bottom=bottom, color=COLORS[s], label=s, edgecolor="white")
    bottom = [b + v for b, v in zip(bottom, h)]
for i, tot in enumerate(bottom):
    ax.text(i, tot + 0.1, f"n={tot}", ha="center", va="bottom", color="#555")
ax.set_xticks(list(x))
ax.set_xticklabels([l for _, l in cats])
ax.set_ylabel("Number of citations")
ax.set_title(f"AtomGPT status on genuine materials references, by reference type (n={len(materials)})")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "materials_by_category.png"), dpi=100)
plt.show()
{c: dict(counts[c]) for c, _ in cats}